In [1]:
# 🏀 March Madness 2025 - Optimized Model with Men & Women Data
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, brier_score_loss

# 📂 Define Data Path
data_path = "data/"

# 🏀 Load Men's & Women's Data
mteams = pd.read_csv(os.path.join(data_path, "MTeams.csv"))
wteams = pd.read_csv(os.path.join(data_path, "WTeams.csv"))

mregular_results = pd.read_csv(os.path.join(data_path, "MRegularSeasonCompactResults.csv"))
wregular_results = pd.read_csv(os.path.join(data_path, "WRegularSeasonCompactResults.csv"))

mtourney_results = pd.read_csv(os.path.join(data_path, "MNCAATourneyCompactResults.csv"))
wtourney_results = pd.read_csv(os.path.join(data_path, "WNCAATourneyCompactResults.csv"))

mtourney_seeds = pd.read_csv(os.path.join(data_path, "MNCAATourneySeeds.csv"))
wtourney_seeds = pd.read_csv(os.path.join(data_path, "WNCAATourneySeeds.csv"))

# ✅ Extract Numeric Seed Values
def extract_seed(seed):
    return int(seed[1:3]) if isinstance(seed, str) and seed[1:3].isdigit() else np.nan

mtourney_seeds["Seed"] = mtourney_seeds["Seed"].apply(extract_seed)
wtourney_seeds["Seed"] = wtourney_seeds["Seed"].apply(extract_seed)

# ✅ Compute Team Statistics
def compute_team_stats(results):
    team_stats = results.groupby(["Season", "WTeamID"]).agg({
        "WScore": ["mean", "sum"],
        "LScore": ["mean", "sum"],
        "NumOT": "sum"
    }).reset_index()

    # Rename columns
    team_stats.columns = ["Season", "TeamID", "AvgWScore", "TotalWScore", "AvgLScore", "TotalLScore", "OTGames"]
    team_stats["ScoreMargin"] = team_stats["TotalWScore"] - team_stats["TotalLScore"]
    team_stats["WinRate"] = team_stats["TotalWScore"] / (team_stats["TotalWScore"] + team_stats["TotalLScore"])

    return team_stats

# Compute Stats for Both Men & Women
mteam_stats = compute_team_stats(mregular_results)
wteam_stats = compute_team_stats(wregular_results)

# 🏀 Prepare Training Data (For Both Men & Women)
def prepare_training_data(results, seeds, team_stats):
    results = results.merge(seeds, left_on=["Season", "WTeamID"], right_on=["Season", "TeamID"], how="left").rename(columns={"Seed": "WSeed"}).drop(columns=["TeamID"])
    results = results.merge(seeds, left_on=["Season", "LTeamID"], right_on=["Season", "TeamID"], how="left").rename(columns={"Seed": "LSeed"}).drop(columns=["TeamID"])
    
    results = results.merge(team_stats.rename(columns={"TeamID": "WTeamID"}), on=["Season", "WTeamID"], how="left")
    results = results.merge(team_stats.rename(columns={"TeamID": "LTeamID"}), on=["Season", "LTeamID"], how="left", suffixes=("_W", "_L"))

    # ✅ Fill Missing Values
    results["WSeed"].fillna(results["WSeed"].median(), inplace=True)
    results["LSeed"].fillna(results["LSeed"].median(), inplace=True)
    results["SeedDiff"] = results["WSeed"] - results["LSeed"]

    # Ensure WinRate & ScoreMargin are present
    results["WinRateDiff"] = results.get("WinRate_W", 0) - results.get("WinRate_L", 0)
    results["ScoreMarginDiff"] = results.get("ScoreMargin_W", 0) - results.get("ScoreMargin_L", 0)

    feature_columns = ["WSeed", "LSeed", "SeedDiff", "WinRateDiff", "ScoreMarginDiff"]
    
    win_features = results[feature_columns].copy()
    win_features["Win"] = 1  

    loss_features = results[feature_columns].copy()
    loss_features["Win"] = 0  

    return pd.concat([win_features, loss_features], ignore_index=True)

# Combine Men's & Women's Data
mtrain_data = prepare_training_data(mtourney_results, mtourney_seeds, mteam_stats)
wtrain_data = prepare_training_data(wtourney_results, wtourney_seeds, wteam_stats)
full_train_data = pd.concat([mtrain_data, wtrain_data], ignore_index=True)

# 🏀 Split and Scale Data
X = full_train_data.drop(columns=["Win"])
y = full_train_data["Win"]
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

# 🏀 Train & Evaluate Models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=500, max_depth=7, min_samples_split=10),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=350, learning_rate=0.04, max_depth=3, min_samples_leaf=3),
    "XGBoost": XGBClassifier(n_estimators=350, learning_rate=0.03, max_depth=5, reg_lambda=1.0, gamma=0.3, subsample=0.8)
}

best_model = None
best_score = float("inf")

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict_proba(X_valid_scaled)[:, 1]
    
    log_loss_score = log_loss(y_valid, y_pred)
    brier_score = brier_score_loss(y_valid, y_pred)
    
    print(f"{name} Log Loss: {log_loss_score:.6f}")
    print(f"{name} Brier Score: {brier_score:.6f}\n{'-'*50}")
    
    if log_loss_score < best_score:
        best_score = log_loss_score
        best_model = model

# 🏀 Cross-Validation Score
cv_scores = cross_val_score(best_model, X_train_scaled, y_train, scoring="neg_log_loss", cv=5)
print(f"Cross-Validation Log Loss: {-np.mean(cv_scores):.6f}")

print("✅ Model Training Completed Successfully!")


C:\Users\JonMa\AppData\Local\Temp\ipykernel_28640\655007421.py:63: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  results["WSeed"].fillna(results["WSeed"].median(), inplace=True)
C:\Users\JonMa\AppData\Local\Temp\ipykernel_28640\655007421.py:64: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as 

Logistic Regression Log Loss: 0.693293
Logistic Regression Brier Score: 0.250073
--------------------------------------------------
Random Forest Log Loss: 0.718784
Random Forest Brier Score: 0.262753
--------------------------------------------------
Gradient Boosting Log Loss: 0.744539
Gradient Boosting Brier Score: 0.274958
--------------------------------------------------
XGBoost Log Loss: 0.782524
XGBoost Brier Score: 0.293860
--------------------------------------------------
Cross-Validation Log Loss: 0.693798
✅ Model Training Completed Successfully!
